# VieNeu-TTS v3 Turbo + Google ADK trên Colab

Notebook này chạy VieNeu OpenAI-compatible server trên GPU Colab, sau đó tạo một Google ADK agent có function tool gọi `POST /v1/audio/speech`. Không cần tunnel vì ADK và VieNeu chạy trong cùng runtime.

Trước khi bắt đầu: chọn **Runtime → Change runtime type → T4 GPU** (hoặc GPU mạnh hơn). Để chạy agent, tạo Gemini API key tại [Google AI Studio](https://aistudio.google.com/apikey), rồi thêm vào **Colab Secrets** với tên `GOOGLE_API_KEY`.

Tiếp theo: lấy file [openai_speech.py](https://drive.google.com/file/d/1ljRk5GymA4Ed73s8HvjGsxcVouYQesZ5/view?usp=drive_link) mới thay file cũ ở folder apps. Và thêm file [test_openai_speech_timing.py](https://drive.google.com/file/d/1WY22C4k27pM1r34A4bailZ8iI1liUGF9/view?usp=drive_link) ở folder tests.

Mục tiêu của việc thay đổi và thêm file mới ở trên để thực hiện yêu cầu về tracing khi trong quá trình tạo âm thanh.

## 1. Kiểm tra GPU và cài công cụ

In [1]:
import os
import shutil
import subprocess
import sys
from pathlib import Path

from IPython.display import clear_output

gpu = subprocess.run(["nvidia-smi"], text=True, capture_output=True)
if gpu.returncode != 0:
    raise RuntimeError("Không tìm thấy GPU. Hãy chọn Runtime → Change runtime type → GPU rồi chạy lại.")

subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "google-adk>=2,<3", "requests"],
    check=True,
)
subprocess.run(
    ["bash", "-lc", "curl -LsSf https://astral.sh/uv/install.sh | sh"],
    check=True,
)
UV = shutil.which("uv") or str(Path.home() / ".local/bin/uv")
if not Path(UV).exists():
    raise RuntimeError("Cài uv không thành công.")

clear_output()
print("GPU đã sẵn sàng:")
print("\n".join(line for line in gpu.stdout.splitlines() if "NVIDIA" in line or "MiB" in line)[:2000])
print(f"uv: {UV}")

GPU đã sẵn sàng:
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
| N/A   47C    P8             11W /   70W |       0MiB /  15360MiB |      0%      Default |
uv: /usr/local/bin/uv


## 2. Tải VieNeu-TTS và cài backend CUDA

Nếu repository đã tồn tại do chạy lại cell, notebook chỉ `git pull --ff-only` thay vì clone lại.

In [2]:
REPO_DIR = Path("/content/VieNeu-TTS")
if REPO_DIR.exists():
    subprocess.run(["git", "pull", "--ff-only"], cwd=REPO_DIR, check=True)
else:
    subprocess.run(
        ["git", "clone", "https://github.com/pnnbao97/VieNeu-TTS.git", str(REPO_DIR)],
        check=True,
    )

subprocess.run([UV, "sync", "--extra", "cuda"], cwd=REPO_DIR, check=True)
subprocess.run(
    [
        UV, "run", "python", "-c",
        "import importlib.metadata, torch; "
        "print('VieNeu', importlib.metadata.version('vieneu')); "
        "print('PyTorch', torch.__version__); "
        "print('CUDA available:', torch.cuda.is_available()); "
        "print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else '-')",
    ],
    cwd=REPO_DIR,
    check=True,
)

CompletedProcess(args=['/usr/local/bin/uv', 'run', 'python', '-c', "import importlib.metadata, torch; print('VieNeu', importlib.metadata.version('vieneu')); print('PyTorch', torch.__version__); print('CUDA available:', torch.cuda.is_available()); print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else '-')"], returncode=0)

## 3. Chạy VieNeu server dưới nền

Khác với `!uv run python -m apps.openai_speech`, `Popen` không giữ cell chạy mãi. Cell chờ tối đa 10 phút để tải model, warm-up CUDA Graph và nhận phản hồi từ `/health`.

In [3]:
import time

import requests

BASE_URL = "http://127.0.0.1:8000"
SERVER_LOG = Path("/content/vieneu_server.log")

# Dừng process do chính notebook tạo nếu cell này được chạy lại.
old_process = globals().get("SERVER_PROCESS")
if old_process is not None and old_process.poll() is None:
    old_process.terminate()
    old_process.wait(timeout=20)

try:
    existing_health = requests.get(f"{BASE_URL}/health", timeout=2)
    existing_health.raise_for_status()
except requests.RequestException:
    existing_health = None

if existing_health is None:
    server_env = os.environ.copy()
    server_env.update({
        "VIENEU_BACKEND": "pytorch",
        "VIENEU_DEVICE": "cuda",
        "VIENEU_MAX_STREAMS": "16",
        "VIENEU_QUEUE": "16",
        "VIENEU_WATERMARK": "1",
        "HOST": "127.0.0.1",
        "PORT": "8000",
    })
    SERVER_LOG_HANDLE = SERVER_LOG.open("w")
    SERVER_PROCESS = subprocess.Popen(
        [UV, "run", "python", "-m", "apps.openai_speech"],
        cwd=REPO_DIR,
        env=server_env,
        stdout=SERVER_LOG_HANDLE,
        stderr=subprocess.STDOUT,
        start_new_session=True,
    )

deadline = time.time() + 600
health = None
last_notice = 0
while time.time() < deadline:
    process = globals().get("SERVER_PROCESS")
    if process is not None and process.poll() is not None:
        tail = SERVER_LOG.read_text(errors="replace")[-5000:]
        raise RuntimeError(f"VieNeu server đã dừng với mã {process.returncode}:\n{tail}")
    try:
        response = requests.get(f"{BASE_URL}/health", timeout=2)
        if response.ok:
            health = response.json()
            break
    except requests.RequestException:
        pass
    if time.time() - last_notice >= 10:
        print("Đang tải model và warm-up server...")
        last_notice = time.time()
    time.sleep(2)

if health is None:
    tail = SERVER_LOG.read_text(errors="replace")[-5000:] if SERVER_LOG.exists() else "Không có log"
    raise TimeoutError(f"Server chưa sẵn sàng sau 10 phút:\n{tail}")

print("VieNeu server sẵn sàng:", health)

Đang tải model và warm-up server...
Đang tải model và warm-up server...
Đang tải model và warm-up server...
Đang tải model và warm-up server...
VieNeu server sẵn sàng: {'status': 'ok', 'backend': 'pytorch', 'max_streams': 16, 'active': 0, 'waiting': 0, 'sample_rate': 48000}


## 4. Kiểm tra VieNeu API trực tiếp

Cell này chưa dùng ADK. Nó giúp xác nhận server, giọng và audio streaming hoạt động trước khi thêm Gemini vào luồng. Response dùng SSE để giữ nguyên ranh giới từng chunk; phần audio bên trong vẫn là PCM và được notebook đóng gói thành WAV để phát. Log hiển thị cách ngắt text, thời gian tạo chunk tại server/GPU và thời điểm chunk đến client.

In [4]:
import base64
import json
import wave
from datetime import datetime

from IPython.display import Audio, display

def stream_vieneu_to_wav(
    text: str, voice: str, output_path: str, max_chars: int = 256
) -> dict:
    start = time.perf_counter()
    first_audio_at = None
    audio_bytes = 0
    audio_chunk_count = 0
    text_chunks = []
    server_generation_ms = 0.0
    request_id = None
    payload = {
        "model": "vieneu-v3-turbo",
        "input": text,
        "voice": voice,
        "response_format": "pcm",
        "stream_format": "sse",
        "sample_rate": 48000,
        "max_chars": max_chars,
    }
    with requests.post(
        f"{BASE_URL}/v1/audio/speech",
        json=payload,
        stream=True,
        timeout=(10, 300),
    ) as response:
        if not response.ok:
            raise RuntimeError(f"VieNeu HTTP {response.status_code}: {response.text[:1000]}")
        request_id = response.headers.get("X-Request-Id")
        with wave.open(output_path, "wb") as wav_file:
            wav_file.setnchannels(1)
            wav_file.setsampwidth(2)
            wav_file.setframerate(48000)
            for line in response.iter_lines(decode_unicode=True):
                if not line or not line.startswith("data:"):
                    continue
                received_at = time.perf_counter()
                received_wall = datetime.now().astimezone()
                event = json.loads(line[5:].lstrip())
                event_type = event.get("type")

                if event_type == "speech.text.chunks":
                    text_chunks = event.get("chunks", [])
                    print(f"\n===== TEXT CHUNKS ({len(text_chunks)}) =====")
                    for item in text_chunks:
                        chunk_text = item["text"].replace("\n", " ")
                        print(
                            f"#{item['index']:>2} | chars={item['characters']:>4} | {chunk_text}"
                        )
                    continue

                if event_type != "speech.audio.delta":
                    continue
                timing = event.get("vieneu")
                if timing is None:
                    raise RuntimeError(
                        "Server chưa hỗ trợ metadata thời gian. Hãy cập nhật "
                        "apps/openai_speech.py rồi khởi động lại cell server."
                    )
                chunk = base64.b64decode(event["audio"], validate=True)
                if first_audio_at is None:
                    first_audio_at = received_at
                    print("\n===== AUDIO CHUNKS =====")
                wav_file.writeframesraw(chunk)
                audio_bytes += len(chunk)
                audio_chunk_count += 1
                server_generation_ms += timing["generation_ms"]
                server_finished = datetime.fromisoformat(timing["server_finished_at"])
                transport_ms = (received_wall - server_finished).total_seconds() * 1000
                client_offset_ms = (received_at - start) * 1000
                print(
                    f"#{timing['index']:>2} | samples={timing['samples']:>6} "
                    f"| audio={timing['audio_ms']:>7.1f} ms "
                    f"| GPU={timing['generation_ms']:>7.1f} ms "
                    f"({timing['server_start_offset_ms']:.1f}→"
                    f"{timing['server_end_offset_ms']:.1f} ms) "
                    f"| server={timing['server_started_at']}→"
                    f"{timing['server_finished_at']} "
                    f"| client={received_wall.isoformat(timespec='milliseconds')} "
                    f"| offset={client_offset_ms:.1f} ms "
                    f"| transport≈{transport_ms:.1f} ms"
                )

    duration = audio_bytes / (48000 * 2)
    total_seconds = time.perf_counter() - start
    rtf = total_seconds / duration if duration else None
    realtime_speed = duration / total_seconds if duration and total_seconds else None
    print("\n===== RESULT =====")
    print(f"Text chunks     : {len(text_chunks)}")
    print(f"Audio chunks    : {audio_chunk_count}")
    print(f"Audio duration  : {duration:.3f} s")
    print(f"Total time      : {total_seconds:.3f} s")
    print(f"GPU chunk time  : {server_generation_ms:.1f} ms")
    print(f"TTFA            : {(first_audio_at - start) * 1000:.1f} ms")
    print(f"RTF             : {rtf:.3f}" if rtf is not None else "RTF             : n/a")
    print(
        f"Realtime speed  : {realtime_speed:.2f}x"
        if realtime_speed is not None else "Realtime speed  : n/a"
    )
    return {
        "request_id": request_id,
        "output_path": output_path,
        "ttfa_ms": round((first_audio_at - start) * 1000, 1) if first_audio_at else None,
        "audio_seconds": round(duration, 3),
        "total_seconds": round(total_seconds, 3),
        "text_chunk_count": len(text_chunks),
        "audio_chunk_count": audio_chunk_count,
        "total_samples": audio_bytes // 2,
        "server_generation_ms": round(server_generation_ms, 1),
        "rtf": round(rtf, 3) if rtf is not None else None,
        "realtime_speed": round(realtime_speed, 2) if realtime_speed is not None else None,
    }

DIRECT_RESULT = stream_vieneu_to_wav(
    "Xin chào. VieNeu server trên Google Colab đã sẵn sàng.",
    "Mai Anh",
    "/content/vieneu_direct_test.wav",
)
print(DIRECT_RESULT)
display(Audio(DIRECT_RESULT["output_path"]))


===== TEXT CHUNKS (1) =====
# 1 | chars=  55 | xin chào, vie neu server trên google colab đã sẵn sàng.

===== AUDIO CHUNKS =====
# 1 | samples=  7680 | audio=  160.0 ms | GPU=  377.9 ms (0.4→378.2 ms) | server=2026-09-17T10:13:47.520+00:00→2026-09-17T10:13:47.898+00:00 | client=2026-09-17T10:13:47.899+00:00 | offset=388.3 ms | transport≈1.8 ms
# 2 | samples=  7680 | audio=  160.0 ms | GPU=  136.2 ms (379.4→515.6 ms) | server=2026-09-17T10:13:47.899+00:00→2026-09-17T10:13:48.035+00:00 | client=2026-09-17T10:13:48.037+00:00 | offset=525.8 ms | transport≈2.3 ms
# 3 | samples= 15360 | audio=  320.0 ms | GPU=  263.2 ms (518.0→781.2 ms) | server=2026-09-17T10:13:48.037+00:00→2026-09-17T10:13:48.301+00:00 | client=2026-09-17T10:13:48.306+00:00 | offset=794.7 ms | transport≈5.3 ms
# 4 | samples= 15360 | audio=  320.0 ms | GPU=  262.2 ms (782.7→1044.9 ms) | server=2026-09-17T10:13:48.302+00:00→2026-09-17T10:13:48.564+00:00 | client=2026-09-17T10:13:48.569+00:00 | offset=1058.0 ms | transport≈5

## 5. Cấu hình Gemini API key cho Google ADK

Khuyến nghị: mở biểu tượng chìa khóa ở thanh bên Colab, tạo secret `GOOGLE_API_KEY` và bật quyền truy cập cho notebook. Nếu chưa có secret, cell sẽ hỏi key ẩn bằng `getpass`; key không được ghi vào notebook.

In [ ]:
from getpass import getpass

try:
    from google.colab import userdata
    google_api_key = userdata.get("GOOGLE_API_KEY")
except Exception:
    google_api_key = None

google_api_key = ""

if not google_api_key:
    google_api_key = getpass("Nhập GOOGLE_API_KEY: ").strip()
if not google_api_key:
    raise ValueError("GOOGLE_API_KEY không được để trống.")

os.environ["GOOGLE_API_KEY"] = google_api_key
os.environ["GOOGLE_GENAI_USE_VERTEXAI"] = "FALSE"
print("Đã cấu hình Gemini API key cho ADK.")

Đã cấu hình Gemini API key cho ADK.


## 6. Tạo VieNeu function tool và Google ADK agent

ADK tự bọc hàm Python trong `tools=[...]` thành `FunctionTool`. Tool chỉ nhận hai tham số để Gemini dễ gọi đúng: nội dung và tên giọng.

In [6]:
import uuid

from google.adk.agents.llm_agent import Agent

LAST_AUDIO_PATH = None

def synthesize_speech(text: str, voice: str = "Mai Anh") -> dict:
    """Tạo giọng nói tiếng Việt/Anh bằng VieNeu và lưu thành WAV.

    Args:
        text: Nội dung cần đọc thành tiếng.
        voice: Tên giọng VieNeu, mặc định là Mai Anh.

    Returns:
        Trạng thái, đường dẫn WAV, TTFA và thời lượng audio.
    """
    global LAST_AUDIO_PATH
    output_path = f"/content/vieneu_adk_{uuid.uuid4().hex[:8]}.wav"
    try:
        result = stream_vieneu_to_wav(text, voice, output_path)
    except Exception as exc:
        return {"status": "error", "message": str(exc)}
    LAST_AUDIO_PATH = output_path
    return {"status": "success", **result}

root_agent = Agent(
    model="gemini-3.6-flash",
    name="vieneu_speech_agent",
    description="Agent tạo file giọng nói bằng VieNeu-TTS.",
    instruction=(
        "Bạn là trợ lý tạo giọng nói. Khi người dùng yêu cầu đọc hoặc tạo audio, "
        "luôn gọi tool synthesize_speech. Giữ nguyên nội dung người dùng muốn đọc; "
        "không tuyên bố thành công nếu tool trả về lỗi. Sau khi tool chạy, báo tên giọng, "
        "đường dẫn file, TTFA và thời lượng audio thật ngắn gọn."
    ),
    tools=[synthesize_speech],
)
print("Đã tạo ADK agent:", root_agent.name)

Đã tạo ADK agent: vieneu_speech_agent


## 7. Chạy agent bằng ADK Runner

Sửa `TEXT_TO_SPEAK` và `VOICE`, sau đó chạy cell. Prompt yêu cầu Gemini truyền nguyên văn nội dung vào VieNeu tool.

In [7]:
from google.adk.runners import Runner
from google.adk.sessions import InMemorySessionService
from google.genai import types

APP_NAME = "vieneu_adk_colab"
USER_ID = "colab_user"
SESSION_ID = f"session_{uuid.uuid4().hex[:8]}"

session_service = InMemorySessionService()
await session_service.create_session(
    app_name=APP_NAME,
    user_id=USER_ID,
    session_id=SESSION_ID,
)
runner = Runner(agent=root_agent, app_name=APP_NAME, session_service=session_service)

async def run_agent(prompt: str) -> str:
    message = types.Content(role="user", parts=[types.Part(text=prompt)])
    final_text = ""
    async for event in runner.run_async(
        user_id=USER_ID,
        session_id=SESSION_ID,
        new_message=message,
    ):
        if event.is_final_response() and event.content:
            final_text = "".join(
                part.text or "" for part in event.content.parts or []
            )
    print("ADK agent:", final_text)
    return final_text

TEXT_TO_SPEAK = (
    "Xin chào các bạn. Đây là bài kiểm tra Google ADK gọi VieNeu-TTS "
    "để tạo âm thanh streaming trên GPU Colab."
)
VOICE = "Mai Anh"
LAST_AUDIO_PATH = None
prompt = (
    f"Hãy dùng tool synthesize_speech với voice={VOICE!r} để đọc nguyên văn "
    f"nội dung nằm giữa thẻ <text> sau: <text>{TEXT_TO_SPEAK}</text>"
)
await run_agent(prompt)

if not LAST_AUDIO_PATH:
    raise RuntimeError("Agent chưa tạo audio. Hãy xem phản hồi phía trên và kiểm tra Gemini quota/API key.")
print("File audio:", LAST_AUDIO_PATH)
display(Audio(LAST_AUDIO_PATH))

/usr/local/lib/python3.13/dist-packages/google/adk/models/llm_request.py:273: UserWarning: [EXPERIMENTAL] feature FeatureName.JSON_SCHEMA_FOR_FUNC_DECL is enabled.
  declaration = tool._get_declaration()



===== TEXT CHUNKS (1) =====
# 1 | chars= 132 | xin chào các bạn. đây là bài kiểm tra google a đê ca gọi vie neu <en>t t s</en> để tạo âm thanh streaming trên <en>g p u</en> colab.

===== AUDIO CHUNKS =====
# 1 | samples=  7680 | audio=  160.0 ms | GPU=  382.1 ms (0.3→382.4 ms) | server=2026-09-17T10:13:59.676+00:00→2026-09-17T10:14:00.058+00:00 | client=2026-09-17T10:14:00.060+00:00 | offset=391.2 ms | transport≈2.4 ms
# 2 | samples=  7680 | audio=  160.0 ms | GPU=  147.8 ms (385.1→532.9 ms) | server=2026-09-17T10:14:00.060+00:00→2026-09-17T10:14:00.208+00:00 | client=2026-09-17T10:14:00.211+00:00 | offset=541.9 ms | transport≈3.1 ms
# 3 | samples= 15360 | audio=  320.0 ms | GPU=  260.3 ms (534.6→794.9 ms) | server=2026-09-17T10:14:00.210+00:00→2026-09-17T10:14:00.470+00:00 | client=2026-09-17T10:14:00.476+00:00 | offset=807.3 ms | transport≈6.5 ms
# 4 | samples= 15360 | audio=  320.0 ms | GPU=  269.1 ms (796.4→1065.5 ms) | server=2026-09-17T10:14:00.472+00:00→2026-09-17T10:14:00.741+

## 8. Kiểm tra log hoặc dừng server

Chạy cell log khi cần xem TTFA/RTF phía server. Chỉ chạy cell dừng server khi đã thử nghiệm xong.

In [8]:
print(SERVER_LOG.read_text(errors="replace")[-5000:] if SERVER_LOG.exists() else "Không có file log.")

/content/VieNeu-TTS/apps/openai_speech.py:188: DeprecationWarning: 
        on_event is deprecated, use lifespan event handlers instead.

        Read more about it in the
        [FastAPI docs for Lifespan Events](https://fastapi.tiangolo.com/advanced/events/).
        
  @app.on_event("startup")
INFO:     Started server process [1895]
INFO:     Waiting for application startup.
2026-09-17 10:13:08,806 INFO vieneu.api: ⏳ loading VieNeu-TTS v3 Turbo (backend=pytorch device=cuda)
2026-09-17 10:13:21,391 INFO Vieneu.V3Turbo: ⏳ Loading VieNeu-TTS v3 Turbo (PyTorch) from: pnnbao-ump/VieNeu-TTS-v3-Turbo/update ...
A new version of the following files was downloaded from https://huggingface.co/OpenMOSS-Team/MOSS-Audio-Tokenizer-Nano:
- configuration_moss_audio_tokenizer.py
. Make sure to double-check they do not contain any added malicious code. To avoid downloading new versions of the code file, you can pin a revision.
A new version of the following files was downloaded from https://huggingf

In [9]:
process = globals().get("SERVER_PROCESS")
if process is not None and process.poll() is None:
    process.terminate()
    process.wait(timeout=20)
    print("Đã dừng VieNeu server.")
else:
    print("Notebook không sở hữu server đang chạy hoặc server đã dừng.")

Đã dừng VieNeu server.
